# BT4012 — Bitcoin transaction classification

Two final submissions: Dynamic Adaptive and Dynamic Plus Guarded.

## Data loading and feature selection

The final classifiers use the 165 supplied numeric transaction features. Labels are separated into known licit, known illicit and unlabelled observations. The graph edge list is not added to the final feature matrix.

In [1]:
from pathlib import Path
import pandas as pd
from dynamic_adaptive import load_data
from dynamic_plus_guarded import validate, generate_both

root = Path.cwd()
data = load_data(root / "data")
print("Training observations:", len(data["train"]))
print("Labelled observations:", int(data["known"].sum()))
print("Illicit observations:", int(data["target"].sum()))
print("Test observations:", len(data["test"]))
print("Number of features:", data["x_train"].shape[1])

Training observations: 141772
Labelled observations: 31235
Illicit observations: 3644
Test observations: 15329
Number of features: 165


## Chronological model evaluation

Validation trains on earlier time steps and evaluates on three later, non-overlapping windows. ROC-AUC measures ranking performance; average precision is also reported because illicit cases are uncommon.
Set `RUN_VALIDATION = True` in the following cell to retrain and evaluate the historical folds. Saved final models are used only for the subsequent final inference.


In [2]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    validation = validate(data)
    validation.to_csv(root / "validation_results.csv", index=False)
    display(validation.round(6))
else:
    print("Validation skipped for inference; set RUN_VALIDATION = True to rerun folds")


Validation skipped for inference; set RUN_VALIDATION = True to rerun folds


## Final saved-model inference and export

The final six-model ensemble is loaded from the saved fitted models in `models/`, then two saved LightGBM specialists produce the bounded Guarded correction. All eight models were trained on the permitted training steps. The scripts also permit full retraining by setting `RETRAIN_FROM_SCRATCH=1`.

In [3]:
dynamic_csv, guarded_csv = generate_both(data, root)
print("Dynamic Adaptive:", len(dynamic_csv), "rows")
print("Plus Guarded:", len(guarded_csv), "rows")
assert dynamic_csv["index"].equals(data["template"]["index"])
assert guarded_csv["index"].equals(data["template"]["index"])
print(dynamic_csv.head())
print(guarded_csv.head())

Using the saved fitted models for final inference
Saved c:\Users\zwjf0\OneDrive - National University of Singapore\4.1 BT4012\kaggle competition\BT4012_Frozen_Models_Reproduction\BT4012\dynamic_adaptive_submission_7.csv (15,329 rows)
Saved c:\Users\zwjf0\OneDrive - National University of Singapore\4.1 BT4012\kaggle competition\BT4012_Frozen_Models_Reproduction\BT4012\dynamic_plus_guarded_8.csv (15,329 rows)
Dynamic Adaptive: 15329 rows
Plus Guarded: 15329 rows
   index    target
0      0  0.000464
1      1  0.008988
2      2  0.005357
3      3  0.000913
4      4  0.000309
   index    target
0      0  0.000468
1      1  0.008886
2      2  0.005393
3      3  0.000914
4      4  0.000310


## Comparison with original submitted predictions

The saved reference files are not inputs to training. Compare them to freshly generated predictions separately.

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

for filename in ("dynamic_adaptive_submission_7.csv", "dynamic_plus_guarded_8.csv"):
    original = pd.read_csv(Path("submitted_reference") / filename)
    reproduced = pd.read_csv(filename)
    assert original["index"].equals(reproduced["index"])
    x = original["target"].to_numpy()
    y = reproduced["target"].to_numpy()
    print(filename, "exact:", np.array_equal(x, y),
          "max_diff:", np.max(np.abs(x-y)),
          "rho:", spearmanr(x, y).statistic,
          "same_order:", np.array_equal(np.argsort(x,kind="stable"), np.argsort(y,kind="stable")))


dynamic_adaptive_submission_7.csv exact: False max_diff: 2.101327266412767e-06 rho: 0.9999999999999998 same_order: True
dynamic_plus_guarded_8.csv exact: False max_diff: 2.0903589031950887e-06 rho: 1.0 same_order: True
